# Experiment 2 (BERT leg) — X → Y vs X + C → Y

The **one-thing test** with a fine-tuned BERT backbone: identical model,
same 60/40 document-level split, same seed (42), 3 epochs, lr 2e-5, batch 16.
The only change between conditions is the addition of the idea-bottleneck
features C (multi-hot over the 8,706-idea space, concatenated to the [CLS]
pooled embedding before the linear head).

- **A (control):** BERT [CLS] → linear head → label
- **B (experiment):** BERT [CLS] + C → linear head → label

Companion result (LR, exp2): A 0.6559/0.6581 → B 0.7541/0.7555 (+9.8pt).
This notebook answers the same question for BERT.

Metrics: accuracy + macro-F1 on the held-out test set.


In [1]:
# 1. Setup + fetch committed inputs
import os
os.chdir('/content')
if not os.path.exists('/content/qsbc'):
    !git clone -q https://github.com/jpeckenpaugh/qsbc.git qsbc
os.chdir('/content/qsbc')
print('cwd:', os.getcwd())
!pip install -q pandas scikit-learn transformers datasets torch accelerate


cwd: /content/qsbc


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

df = pd.read_csv('results/exp1/samples_2858.csv')
df = df.drop_duplicates(subset=['sentence_id']).reset_index(drop=True)
ideas = pd.read_csv('results/exp1/sample_ideas.csv')

grp = ideas.groupby('sentence_id')['idea_id'].apply(list).to_dict()
df['ideas'] = df['sentence_id'].map(grp).fillna('').apply(
    lambda x: x if isinstance(x, list) else [])
print('samples:', len(df), '| with >=1 idea:', int((df['ideas'].str.len() > 0).sum()))
print('idea space size:', ideas['idea_id'].nunique())


samples: 2858 | with >=1 idea: 2782
idea space size: 8706


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

SEED = 42
docs = df['doc_key'].unique()
train_docs, test_docs = train_test_split(docs, test_size=0.4, random_state=SEED)
train = df[df['doc_key'].isin(train_docs)].reset_index(drop=True)
test  = df[df['doc_key'].isin(test_docs)].reset_index(drop=True)
print('doc overlap:', len(set(train_docs) & set(test_docs)))
print('train:', len(train), '| test:', len(test))
print('train classes:', train['label'].nunique(), '| test classes:', test['label'].nunique())

mlb = MultiLabelBinarizer()
mlb.fit(list(train['ideas']) + list(test['ideas']))
Ctr = np.asarray(mlb.transform(list(train['ideas'])), dtype=np.float32)
Cte = np.asarray(mlb.transform(list(test['ideas'])), dtype=np.float32)
print('C dims:', Ctr.shape)


doc overlap: 0
train: 1748 | test: 1110
train classes: 12 | test classes: 12
C dims: (1748, 8706)


In [4]:
classes = sorted(set(train['label']) | set(test['label']))
id2label = {i: c for i, c in enumerate(classes)}
label2id = {c: i for i, c in id2label.items()}

from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def make_ds(frame, Cmat):
    ds = Dataset.from_dict({
        'sentence': frame['sentence'],
        'label': frame['label'].map(label2id),
    })
    def tok(batch):
        return tokenizer(batch['sentence'], truncation=True, padding='max_length',
                        max_length=128)
    ds = ds.map(tok, batched=True)
    ds = ds.add_column('c_features', [Cmat[i] for i in range(len(Cmat))])
    return ds

train_ds = make_ds(train, Ctr)
test_ds  = make_ds(test, Cte)

cols = ['input_ids', 'attention_mask', 'label', 'c_features']
train_ds.set_format('torch', columns=cols)
test_ds.set_format('torch', columns=cols)
print('train_ds:', train_ds.num_rows, '| test_ds:', test_ds.num_rows)


Map:   0%|          | 0/1748 [00:00<?, ? examples/s]

Map:   0%|          | 0/1110 [00:00<?, ? examples/s]

train_ds: 1748 | test_ds: 1110


In [5]:
from transformers import AutoModel, AutoModelForSequenceClassification, Trainer, TrainingArguments
from transformers.modeling_outputs import SequenceClassifierOutput
import torch.nn as nn
import torch

class BertPlusC(nn.Module):
    """BERT [CLS] pooled + C multi-hot -> linear head. C block stays linear."""
    def __init__(self, model_name, n_ideas, n_labels):
        super().__init__()
        self.num_labels = n_labels
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size + n_ideas, n_labels)

    def forward(self, input_ids, attention_mask, c_features, labels=None):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.pooler_output
        x = torch.cat([pooled, c_features.float()], dim=1)
        logits = self.classifier(self.dropout(x))

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits
        )

def make_args(out_dir):
    return TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=3,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        seed=SEED,
        report_to=[],
        save_strategy='no',
        logging_steps=50,
    )


In [6]:
# A (control): standard BERT sequence classifier
modelA = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=len(classes), id2label=id2label, label2id=label2id)
trainerA = Trainer(model=modelA, args=make_args('/content/bertA'),
                   train_dataset=train_ds, eval_dataset=test_ds)
trainerA.train()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,2.391435
100,2.151858
150,1.733559
200,1.551194
250,1.291751
300,1.210536


TrainOutput(global_step=330, training_loss=1.6709748470421992, metrics={'train_runtime': 28.0878, 'train_samples_per_second': 186.7, 'train_steps_per_second': 11.749, 'total_flos': 344969564221440.0, 'train_loss': 1.6709748470421992, 'epoch': 3.0})

In [7]:
# B (experiment): BERT + C multi-hot
modelB = BertPlusC('bert-base-uncased', n_ideas=Ctr.shape[1], n_labels=len(classes))
trainerB = Trainer(model=modelB, args=make_args('/content/bertB'),
                   train_dataset=train_ds, eval_dataset=test_ds)
trainerB.train()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
50,2.465123
100,2.336755
150,2.157593
200,2.041299
250,1.911261
300,1.844567


TrainOutput(global_step=330, training_loss=2.0984892411665483, metrics={'train_runtime': 27.4838, 'train_samples_per_second': 190.804, 'train_steps_per_second': 12.007, 'total_flos': 0.0, 'train_loss': 2.0984892411665483, 'epoch': 3.0})

In [8]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def score(trainer, ds, name):
    out = trainer.predict(ds)
    p = [id2label[int(x)] for x in np.argmax(out.predictions, axis=1)]
    acc = accuracy_score(test['label'], p)
    f1 = f1_score(test['label'], p, average='macro')
    print(f'{name:14s} acc={acc:.4f}  macro-F1={f1:.4f}')
    return acc, f1

accA, f1A = score(trainerA, test_ds, 'A: X only')
accB, f1B = score(trainerB, test_ds, 'B: X + C')
print('\nDelta (B - A):  acc %+.4f   macro-F1 %+.4f' % (accB - accA, f1B - f1A))


A: X only      acc=0.6468  macro-F1=0.6394


B: X + C       acc=0.4874  macro-F1=0.4433

Delta (B - A):  acc -0.1595   macro-F1 -0.1960


In [9]:
import pandas as pd
rows = [
    ['BERT A: X -> Y (control)', accA, f1A],
    ['BERT B: X + C -> Y (experiment)', accB, f1B],
    ['LR A: X -> Y (exp2)', 0.655856, 0.658133],
    ['LR B: X + C -> Y (exp2)', 0.754054, 0.755501],
]
print(pd.DataFrame(rows, columns=['Condition', 'Accuracy', 'Macro-F1']).to_string(index=False))


                      Condition  Accuracy  Macro-F1
       BERT A: X -> Y (control)  0.646847  0.639354
BERT B: X + C -> Y (experiment)  0.487387  0.443345
            LR A: X -> Y (exp2)  0.655856  0.658133
        LR B: X + C -> Y (exp2)  0.754054  0.755501
